# Recommendation System for Movies

This notebook contains the Data Cleaning for the Capstone Project regarding a Recommendation System for Movies. 
- This notebook serves to read and join the tables from different data set. 
- The reader is expected to run the codes and ty to understand the contents of the data files. 



## Loading Packages

In [ ]:
import sys
# data analysis stack
import numpy as np
import pandas as pd
import os

# data visualization stack
import matplotlib.pyplot as plt
%matplotlib inline
# import seaborn as sns
# sns.set_style('whitegrid')

# miscellaneous
import warnings
warnings.filterwarnings("ignore")

# Loading Datasets 
First dataset is from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata/data

In [ ]:
df_links = pd.read_csv(r'C:\Users\shabn\neuefische\ds-capstone-recommendation\data\ml-latest-small\links.csv')
df_movies = pd.read_csv(r'C:\Users\shabn\neuefische\ds-capstone-recommendation\data\ml-latest-small\movies.csv')
df_ratings = pd.read_csv(r'C:\Users\shabn\neuefische\ds-capstone-recommendation\data\ml-latest-small\ratings.csv')
df_tags = pd.read_csv(r'C:\Users\shabn\neuefische\ds-capstone-recommendation\data\ml-latest-small\tags.csv')




# Data Exploration

In [ ]:
# Display information and first few rows for each table (links)
print("Links Table")
print(df_links.info())
print(df_links.head())


In [ ]:
# Display information and first few rows for each table (movies)
print("Movies Table")
print(df_movies.info())
print(df_movies.head())


In [ ]:
# Display information and first few rows for each table (ratings)
print("Ratings Table")
print(df_ratings.info())
print(df_ratings.head())


In [ ]:
# Display information and first few rows for each table (tags)
print("Tags Table")
print(df_tags.info())
print(df_tags.head())


# Data cleaning 

In [ ]:
# Convert timestamp to datetime in table rankings
if 'timestamp' in df_ratings.columns:
    df_ratings['timestamp'] = pd.to_datetime(df_ratings['timestamp'], unit='s')

print(df_ratings.head())


In [ ]:
# Convert timestamp to datetime in table tags
if 'timestamp' in df_tags.columns:
    df_tags['timestamp'] = pd.to_datetime(df_tags['timestamp'], unit='s')

print(df_tags.head())

# Decide on a Model type

## Preparation for User-Item Matrix for collaborative filtering models


In [ ]:
user_item_matrix = df_ratings.pivot_table(index='userId', columns='movieId', values='rating')

In [ ]:
# Install and Import Surprise
!pip install scikit-surprise

### Importing the respective packages

In [ ]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

In [ ]:
# Prepare the data for Surprise
reader = Reader(rating_scale=(df_ratings['rating'].min(), df_ratings['rating'].max()))
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)


In [ ]:
# Train-test split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# Train the SVD model
model = SVD()
model.fit(trainset)


In [ ]:
# Make predictions and evaluate
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print("RMSE: {rmse}")

#### User-Based Collaborative Filtering using Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Fill NaNs with zeros for similarity calculation
user_item_matrix_filled = user_item_matrix.fillna(0)

# Calculate cosine similarity
user_similarity = cosine_similarity(user_item_matrix_filled)


#### Content-Based Filtering Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Vectorize the genres column
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres'])

# Calculate similarity
movie_similarity = cosine_similarity(tfidf_matrix)

## Hybrid Model